# Notebook 04: Post-Transplant Outcomes Model
## TransPlan Validation & Reproducibility Pack (Phase 4 M2)

This notebook documents the **post-transplant outcomes model** introduced in Phase 4 Milestone 2. While earlier notebooks focus on *getting* a transplant (wait times, competing risks), this one asks: **what happens after transplant?**

We cover:

1. **Model explanation** — SRTR PSR C-series tables, risk-adjusted Bayesian hierarchical estimates
2. **National baselines** — graft and patient survival at 1-year and 3-year for all 6 organs
3. **City variation heatmap** — 1-year graft survival across 22 cities and 6 organs
4. **Hazard ratio forest plot** — kidney center-level HR with 95% credible intervals, colored by SRTR performance rating
5. **Compound success metric** — P(transplant within 24 months) x P(1-year graft survival)
6. **Performance rating distribution** — how many centers are rated better / as expected / worse per organ
7. **Pancreas handling** — no adult graft survival data; fallback to patient survival
8. **Summary and limitations**

### Data Source
SRTR Program-Specific Reports (PSR), January 2025 release. Tables C5-C12 (graft survival) and C11-C20 (patient survival). Adult (18+) risk-adjusted Bayesian hierarchical estimates. Performance ratings derived from 1-year hazard ratio 95% credible intervals vs. expected.

### References
- SRTR PSR Technical Methods: https://www.srtr.org/about-the-data/technical-methods-for-the-program-specific-reports/
- Bayesian hierarchical modeling for center effects: Christiansen & Morris, *JASA*, 1997
- Compound success metric design: TransPlan ADR-021 (Phase 4 design document)

In [ ]:
# --- Setup: path configuration, imports, reproducibility seed ---
import sys, os, json, hashlib
from pathlib import Path

# Add backend/ to Python path so we can import TransPlan services
REPO_ROOT = Path(os.getcwd()).parent if "notebooks" in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(REPO_ROOT / "backend"))
os.chdir(REPO_ROOT / "backend")  # So config.py DATA_DIR resolves correctly

# Initialize backend data loader (required before importing services)
from services.data_loader import load_all
load_all()

import numpy as np
import pandas as pd
import scipy
import matplotlib
matplotlib.use("Agg")  # Non-interactive backend for headless execution
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

# Reproducibility
RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

# Style
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 120

# Output directory for saved figures
FIGURES_DIR = REPO_ROOT / "notebooks" / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

# Load post-transplant outcomes data and log hash for reproducibility
data_path = REPO_ROOT / "data" / "post-transplant-outcomes.json"
with open(data_path) as f:
    outcomes_raw = json.load(f)

file_hash = hashlib.sha256(data_path.read_bytes()).hexdigest()[:12]
print(f"Data file: {data_path.name}")
print(f"SHA-256 (first 12): {file_hash}")
print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}, SciPy: {scipy.__version__}, Pandas: {pd.__version__}")
print(f"Matplotlib: {matplotlib.__version__}, Seaborn: {sns.__version__}")
print(f"RNG seed: {RNG_SEED}")

# Constants
ORGANS = ["kidney", "liver", "heart", "lung", "pancreas", "intestine"]
ALL_CITIES = [
    "Pittsburgh", "Baltimore", "Philadelphia", "New York", "Minneapolis",
    "Madison", "Chicago", "Cleveland", "St. Louis", "Indianapolis",
    "Omaha", "Rochester", "Nashville", "Durham", "Miami",
    "Dallas", "Houston", "Portland", "Seattle", "San Francisco",
    "Los Angeles", "Palo Alto",
]
city_outcomes = outcomes_raw["city_outcomes"]
print(f"\nLoaded outcomes for {len(city_outcomes)} cities, {len(ORGANS)} organs")
print(f"Source: {outcomes_raw['_meta']['source']}")

## 1. Model Explanation

### What SRTR PSR C-Series Tables Measure

The Scientific Registry of Transplant Recipients (SRTR) publishes **Program-Specific Reports** (PSR) for every US transplant center. The C-series tables report post-transplant outcomes:

- **Tables C5-C12**: Graft survival (is the transplanted organ still functioning?)
- **Tables C11-C20**: Patient survival (is the patient still alive?)

Both are reported at **1-year** and **3-year** post-transplant horizons.

### Risk-Adjusted Bayesian Hierarchical Estimates

Raw survival rates would be misleading because centers treat different patient populations. SRTR uses a **Bayesian hierarchical model** that:

1. **Risk-adjusts** for recipient age, diagnosis, donor type, and other clinical factors
2. **Shrinks** extreme estimates toward the national mean (small-volume centers get pulled toward average)
3. Produces **hazard ratios** (HR) with 95% credible intervals comparing each center to expected

A hazard ratio < 1.0 means the center has *lower* risk of graft failure/death than expected; HR > 1.0 means *higher* risk.

### Performance Ratings

SRTR classifies each center based on the 95% credible interval of the 1-year hazard ratio:
- **Better than expected**: upper bound of 95% CI < 1.0
- **As expected**: CI includes 1.0
- **Worse than expected**: lower bound of 95% CI > 1.0

### The Compound Success Metric

TransPlan combines wait-time probability with post-transplant outcomes into a single metric:

$$\text{Compound Success} = P(\text{transplant within 24 months}) \times P(\text{1-year graft survival})$$

This captures the full patient journey: not just *getting* a transplant, but getting one that *works*. A center with short waits but poor outcomes may score worse than one with moderate waits and excellent survival.

In [ ]:
# --- 2. National Baselines Table ---
# Extract national-level graft and patient survival for all 6 organs

baseline_rows = []
for organ in ORGANS:
    nat = outcomes_raw.get(organ, {})
    baseline_rows.append({
        "Organ": organ.title(),
        "Graft Survival 1yr (%)": nat.get("national_graft_survival_1yr"),
        "Graft Survival 3yr (%)": nat.get("national_graft_survival_3yr"),
        "Patient Survival 1yr (%)": nat.get("national_patient_survival_1yr"),
        "Patient Survival 3yr (%)": nat.get("national_patient_survival_3yr"),
    })

baselines_df = pd.DataFrame(baseline_rows).set_index("Organ")

print("=" * 75)
print("NATIONAL POST-TRANSPLANT SURVIVAL BASELINES (SRTR January 2025)")
print("=" * 75)
print()
print(baselines_df.to_string(na_rep="N/A*"))
print()
print("* Pancreas graft survival is N/A: SRTR does not report adult pancreas")
print("  graft survival separately. TransPlan falls back to patient survival")
print("  for the compound success metric.")
print()

# Grouped bar chart of national baselines
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(ORGANS))
width = 0.2
labels = ["Graft 1yr", "Graft 3yr", "Patient 1yr", "Patient 3yr"]
colors = ["#1f77b4", "#aec7e8", "#2ca02c", "#98df8a"]

for i, (col, color) in enumerate(zip(baselines_df.columns, colors)):
    vals = baselines_df[col].values.astype(float)
    # Replace NaN with 0 for plotting (pancreas graft)
    vals_plot = np.nan_to_num(vals, nan=0.0)
    bars = ax.bar(x + i * width, vals_plot, width, label=labels[i], color=color,
                  edgecolor="white", linewidth=0.5)
    # Annotate values
    for j, (v, v_orig) in enumerate(zip(vals_plot, vals)):
        if np.isnan(v_orig):
            ax.text(x[j] + i * width, 2, "N/A", ha="center", fontsize=8,
                    fontweight="bold", color="red", rotation=90)
        else:
            ax.text(x[j] + i * width, v + 0.5, f"{v:.1f}", ha="center",
                    fontsize=8, fontweight="bold", rotation=0)

ax.set_ylabel("Survival Rate (%)", fontsize=12)
ax.set_title("National Post-Transplant Survival Baselines by Organ\n(SRTR PSR, January 2025 Release)",
             fontsize=14, fontweight="bold")
ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels([o.title() for o in ORGANS], fontsize=11)
ax.set_ylim(0, 105)
ax.legend(loc="lower left", fontsize=10, framealpha=0.9)
ax.axhline(90, color="gray", linestyle=":", alpha=0.3, linewidth=1)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "04-national-baselines.png", bbox_inches="tight", dpi=150)
plt.show()
print("\nSaved: figures/04-national-baselines.png")

In [ ]:
# --- 3. City Variation Heatmap: 1-Year Graft Survival ---
# Build a matrix of graft_survival_1yr across 22 cities x 6 organs.
# Pancreas has no graft survival — we show patient_survival_1yr as a fallback (annotated).

heatmap_rows = []
for city in ALL_CITIES:
    row = {"City": city}
    for organ in ORGANS:
        city_data = city_outcomes.get(city, {}).get(organ, {})
        gs = city_data.get("graft_survival_1yr")
        if gs is not None:
            row[organ.title()] = gs
        else:
            # Fallback to patient survival (pancreas case)
            ps = city_data.get("patient_survival_1yr")
            row[organ.title()] = ps  # May still be None if no data at all
    heatmap_rows.append(row)

heatmap_df = pd.DataFrame(heatmap_rows).set_index("City")

# National baselines for reference annotation
nat_refs = {}
for organ in ORGANS:
    nat = outcomes_raw.get(organ, {})
    gs = nat.get("national_graft_survival_1yr")
    if gs is None:
        gs = nat.get("national_patient_survival_1yr")  # pancreas fallback
    nat_refs[organ.title()] = gs

fig, ax = plt.subplots(figsize=(11, 14))

# Custom annotation: show value and bold cells that are NaN
annot_df = heatmap_df.copy()
sns.heatmap(heatmap_df, annot=True, fmt=".1f", cmap="RdYlGn", center=92,
            vmin=70, vmax=100, linewidths=0.5, linecolor="white", ax=ax,
            cbar_kws={"label": "1-Year Survival Rate (%)", "shrink": 0.6},
            mask=heatmap_df.isna())

# Mark NaN cells with "---"
for i in range(heatmap_df.shape[0]):
    for j in range(heatmap_df.shape[1]):
        if pd.isna(heatmap_df.iloc[i, j]):
            ax.text(j + 0.5, i + 0.5, "---", ha="center", va="center",
                    fontsize=9, color="gray", fontstyle="italic")

# Add national baseline markers on the column headers
for j, organ in enumerate([o.title() for o in ORGANS]):
    ref = nat_refs.get(organ)
    if ref is not None:
        suffix = "*" if organ == "Pancreas" else ""
        ax.text(j + 0.5, -0.3, f"Nat: {ref:.1f}{suffix}", ha="center",
                fontsize=8, color="#555555", fontstyle="italic")

ax.set_title("1-Year Graft Survival (%) by City and Organ\n"
             "Green = higher survival, Red = lower survival\n"
             "(Pancreas column shows patient survival*; --- = no program)",
             fontsize=13, fontweight="bold")
ax.set_ylabel("")
ax.set_xlabel("")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "04-city-graft-survival-heatmap.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: figures/04-city-graft-survival-heatmap.png")
print(f"\nCities with intestine data: "
      f"{sum(1 for c in ALL_CITIES if city_outcomes.get(c, {}).get('intestine'))}")
print("(Most cities lack intestine transplant programs — this is expected)")

In [ ]:
# --- 4. Hazard Ratio Forest Plot: Kidney ---
# For kidney, show the 1-year graft hazard ratio with 95% credible intervals
# for all 22 cities, colored by SRTR performance rating.

RATING_COLORS = {
    "better_than_expected": "#2ca02c",  # green
    "as_expected": "#1f77b4",           # blue
    "worse_than_expected": "#d62728",   # red
    "insufficient_data": "#999999",     # gray
}
RATING_LABELS = {
    "better_than_expected": "Better than expected",
    "as_expected": "As expected",
    "worse_than_expected": "Worse than expected",
    "insufficient_data": "Insufficient data",
}

# Collect kidney HR data
kidney_hr_data = []
for city in ALL_CITIES:
    kd = city_outcomes.get(city, {}).get("kidney", {})
    hr = kd.get("graft_hr_1yr")
    ci = kd.get("graft_hr_1yr_ci")
    rating = kd.get("performance_rating", "as_expected")
    n = kd.get("n_1yr", 0)
    if hr is not None and ci is not None:
        kidney_hr_data.append({
            "city": city,
            "hr": hr,
            "ci_lo": ci[0],
            "ci_hi": ci[1],
            "rating": rating,
            "n_1yr": n,
        })

# Sort by HR (ascending — best outcomes first)
kidney_hr_data.sort(key=lambda x: x["hr"])

fig, ax = plt.subplots(figsize=(10, 12))

y_positions = range(len(kidney_hr_data))
for i, d in enumerate(kidney_hr_data):
    color = RATING_COLORS[d["rating"]]
    # Horizontal error bar
    ax.errorbar(d["hr"], i, xerr=[[d["hr"] - d["ci_lo"]], [d["ci_hi"] - d["hr"]]],
                fmt="o", color=color, ecolor=color, elinewidth=2, capsize=4,
                markersize=7, markeredgecolor="white", markeredgewidth=0.5, zorder=3)
    # City label with volume
    ax.text(-0.05, i, f"{d['city']} (n={d['n_1yr']})", ha="right", va="center",
            fontsize=9, color=color, fontweight="bold" if d["rating"] != "as_expected" else "normal")

# Reference line at HR = 1.0 (expected)
ax.axvline(1.0, color="black", linestyle="-", linewidth=1.5, alpha=0.7, zorder=1)
ax.axvspan(0, 1.0, alpha=0.04, color="green", zorder=0)
ax.axvspan(1.0, 3.0, alpha=0.04, color="red", zorder=0)
ax.text(0.6, len(kidney_hr_data) + 0.3, "Lower risk", fontsize=10,
        color="#2ca02c", fontstyle="italic", ha="center")
ax.text(1.5, len(kidney_hr_data) + 0.3, "Higher risk", fontsize=10,
        color="#d62728", fontstyle="italic", ha="center")

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=8, label=l)
    for rating, (c, l) in {
        k: (RATING_COLORS[k], RATING_LABELS[k]) for k in RATING_COLORS
        if any(d["rating"] == k for d in kidney_hr_data)
    }.items()
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=10, framealpha=0.9,
          title="SRTR Performance Rating")

ax.set_yticks(list(y_positions))
ax.set_yticklabels([""] * len(kidney_hr_data))  # Labels placed manually
ax.set_xlabel("1-Year Graft Hazard Ratio (95% Credible Interval)", fontsize=12)
ax.set_title("Kidney 1-Year Graft Hazard Ratio by Center\n"
             "(HR < 1.0 = lower graft failure risk than expected)",
             fontsize=14, fontweight="bold")
ax.set_xlim(0, max(d["ci_hi"] for d in kidney_hr_data) + 0.3)
ax.invert_yaxis()

plt.tight_layout()
fig.savefig(FIGURES_DIR / "04-kidney-hr-forest-plot.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: figures/04-kidney-hr-forest-plot.png")

# Summary statistics
hrs = [d["hr"] for d in kidney_hr_data]
print(f"\nKidney graft HR range: {min(hrs):.3f} - {max(hrs):.3f}")
print(f"Median HR: {np.median(hrs):.3f}")
n_better = sum(1 for d in kidney_hr_data if d["rating"] == "better_than_expected")
n_worse = sum(1 for d in kidney_hr_data if d["rating"] == "worse_than_expected")
print(f"Performance: {n_better} better, {len(kidney_hr_data) - n_better - n_worse} as expected, {n_worse} worse")

In [ ]:
# --- 5. Compound Success Metric ---
# P(transplant within 24mo) x P(1yr graft survival)
# We use the backend service to compute this for all cities with kidney.
# For demonstration, we use a fixed P(transplant 24mo) from the wait-time model
# for a typical kidney A+ / cPRA 0 patient.

from services.distributions import get_wait_time_distribution
from services.outcomes import compute_compound_success, get_graft_survival_1yr, get_patient_survival_1yr

# Compute P(transplant <= 24mo) for kidney A+ cPRA 0 at each city
compound_data = []
for city in ALL_CITIES:
    dist = get_wait_time_distribution("kidney", "A+", city, cpra=0)
    p_transplant_24 = dist.cdf(24)

    # Get graft survival (falls back to patient survival for pancreas)
    gs = get_graft_survival_1yr("kidney", city)
    compound = compute_compound_success(p_transplant_24, gs)

    compound_data.append({
        "City": city,
        "P(transplant 24mo)": p_transplant_24,
        "Graft Survival 1yr (%)": gs,
        "Compound Success": compound,
    })

compound_df = pd.DataFrame(compound_data).sort_values("Compound Success", ascending=False)

# Print top and bottom 5
print("=" * 70)
print("COMPOUND SUCCESS: P(transplant 24mo) x P(1yr graft survival)")
print("Patient profile: Kidney, A+, cPRA 0")
print("=" * 70)
print()
print("--- TOP 5 ---")
for _, row in compound_df.head(5).iterrows():
    print(f"  {row['City']:20s}  P(tx)={row['P(transplant 24mo)']:.1%}  "
          f"GS={row['Graft Survival 1yr (%)']:.1f}%  "
          f"Compound={row['Compound Success']:.1%}")
print()
print("--- BOTTOM 5 ---")
for _, row in compound_df.tail(5).iterrows():
    print(f"  {row['City']:20s}  P(tx)={row['P(transplant 24mo)']:.1%}  "
          f"GS={row['Graft Survival 1yr (%)']:.1f}%  "
          f"Compound={row['Compound Success']:.1%}")

# Visualization: scatter plot of P(transplant) vs graft survival, size = compound
fig, ax = plt.subplots(figsize=(12, 8))

x_vals = compound_df["P(transplant 24mo)"].values * 100
y_vals = compound_df["Graft Survival 1yr (%)"].values
c_vals = compound_df["Compound Success"].values
cities = compound_df["City"].values

scatter = ax.scatter(x_vals, y_vals, c=c_vals, cmap="RdYlGn", s=120,
                     edgecolors="white", linewidth=1, vmin=0.2, vmax=0.8, zorder=3)

# Label every city
for i, city in enumerate(cities):
    # Offset labels to reduce overlap
    offset_x = 1.0 if x_vals[i] < np.median(x_vals) else -1.0
    offset_y = 0.3
    ax.annotate(city, (x_vals[i], y_vals[i]), fontsize=7.5,
                xytext=(offset_x * 8, offset_y * 8), textcoords="offset points",
                ha="left" if offset_x > 0 else "right", va="bottom",
                arrowprops=dict(arrowstyle="-", color="gray", alpha=0.3, lw=0.5))

# Iso-compound curves (constant compound success lines)
for cs_level in [0.3, 0.4, 0.5, 0.6]:
    p_tx_range = np.linspace(10, 95, 200)
    gs_required = (cs_level / (p_tx_range / 100)) * 100
    mask = (gs_required >= 85) & (gs_required <= 100)
    ax.plot(p_tx_range[mask], gs_required[mask], "--", color="gray", alpha=0.3, linewidth=1)
    # Label the iso-curve
    valid_x = p_tx_range[mask]
    valid_y = gs_required[mask]
    if len(valid_x) > 0:
        ax.text(valid_x[-1] + 0.5, valid_y[-1], f"CS={cs_level:.0%}",
                fontsize=7, color="gray", alpha=0.6)

cbar = fig.colorbar(scatter, ax=ax, shrink=0.7, label="Compound Success")
ax.set_xlabel("P(Transplant within 24 months) (%)", fontsize=12)
ax.set_ylabel("1-Year Graft Survival (%)", fontsize=12)
ax.set_title("Compound Success Metric: Kidney (A+, cPRA 0)\n"
             "Dashed lines = iso-compound-success curves",
             fontsize=14, fontweight="bold")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "04-compound-success-scatter.png", bbox_inches="tight", dpi=150)
plt.show()
print("\nSaved: figures/04-compound-success-scatter.png")

In [ ]:
# --- 6. Performance Rating Distribution ---
# Count how many cities receive each SRTR performance rating, per organ.

rating_order = ["better_than_expected", "as_expected", "worse_than_expected", "insufficient_data"]
rating_display = {
    "better_than_expected": "Better",
    "as_expected": "As Expected",
    "worse_than_expected": "Worse",
    "insufficient_data": "Insufficient Data",
}

rating_counts = {organ: {r: 0 for r in rating_order} for organ in ORGANS}
for city in ALL_CITIES:
    for organ in ORGANS:
        city_data = city_outcomes.get(city, {}).get(organ, {})
        rating = city_data.get("performance_rating")
        if rating and rating in rating_order:
            rating_counts[organ][rating] += 1

# Build DataFrame for stacked bar chart
rating_df = pd.DataFrame({
    rating_display[r]: [rating_counts[organ][r] for organ in ORGANS]
    for r in rating_order
}, index=[o.title() for o in ORGANS])

print("=== SRTR Performance Rating Distribution (1-Year Graft HR) ===\n")
print(rating_df.to_string())
print()

# Total programs per organ
for organ in ORGANS:
    total = sum(rating_counts[organ].values())
    print(f"  {organ.title():12s}: {total} programs reporting")

# Stacked horizontal bar chart
fig, ax = plt.subplots(figsize=(12, 5))

bar_colors = ["#2ca02c", "#1f77b4", "#d62728", "#999999"]
rating_df.plot.barh(stacked=True, ax=ax, color=bar_colors, edgecolor="white", linewidth=0.5)

ax.set_xlabel("Number of Centers", fontsize=12)
ax.set_ylabel("")
ax.set_title("SRTR Performance Rating Distribution by Organ\n"
             "(Based on 1-year graft/patient hazard ratio 95% credible interval)",
             fontsize=14, fontweight="bold")
ax.legend(loc="lower right", fontsize=10, framealpha=0.9, title="Performance Rating")

# Annotate counts on each segment
for container_idx, container in enumerate(ax.containers):
    for bar in container:
        width = bar.get_width()
        if width > 0:
            ax.text(bar.get_x() + width / 2, bar.get_y() + bar.get_height() / 2,
                    f"{int(width)}", ha="center", va="center", fontsize=10,
                    fontweight="bold", color="white" if container_idx < 3 else "black")

ax.set_xlim(0, 25)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "04-performance-rating-distribution.png", bbox_inches="tight", dpi=150)
plt.show()
print("\nSaved: figures/04-performance-rating-distribution.png")

In [ ]:
# --- 7. Pancreas Handling: No Adult Graft Survival ---
# Demonstrate that pancreas lacks graft survival and TransPlan falls back to patient survival.

print("=" * 70)
print("PANCREAS: GRAFT vs PATIENT SURVIVAL AVAILABILITY")
print("=" * 70)
print()

# National baselines
nat_pancreas = outcomes_raw.get("pancreas", {})
print(f"National graft survival 1yr:   {nat_pancreas.get('national_graft_survival_1yr')}")
print(f"National graft survival 3yr:   {nat_pancreas.get('national_graft_survival_3yr')}")
print(f"National patient survival 1yr: {nat_pancreas.get('national_patient_survival_1yr')}")
print(f"National patient survival 3yr: {nat_pancreas.get('national_patient_survival_3yr')}")
print()

# Per-city: show that no city has graft_survival_1yr for pancreas
print("City-level pancreas data (showing graft_survival_1yr availability):")
print("-" * 55)
pancreas_rows = []
for city in ALL_CITIES:
    pd_data = city_outcomes.get(city, {}).get("pancreas", {})
    if pd_data:
        gs = pd_data.get("graft_survival_1yr")
        ps = pd_data.get("patient_survival_1yr")
        n = pd_data.get("n_1yr", 0)
        rating = pd_data.get("performance_rating", "N/A")
        pancreas_rows.append({
            "City": city,
            "Graft Surv 1yr": gs if gs is not None else "N/A",
            "Patient Surv 1yr": ps,
            "n (1yr)": n,
            "Rating": rating,
        })
        ps_str = f"{ps:6.1f}%" if ps is not None else "   N/A"
        print(f"  {city:20s}  GS={str(gs) if gs is not None else 'N/A':>5s}  PS={ps_str}  n={n:3d}  ({rating})")
    else:
        print(f"  {city:20s}  (no pancreas program)")

print()
print(f"Cities with pancreas programs: {len(pancreas_rows)}")
print(f"Cities with graft survival data: "
      f"{sum(1 for r in pancreas_rows if r['Graft Surv 1yr'] != 'N/A')}")
print()
print("CONCLUSION: All pancreas entries have graft_survival_1yr = null.")
print("TransPlan's compound success metric uses patient_survival_1yr as a")
print("proxy, with a note: 'Uses patient survival (graft survival unavailable")
print("for this organ)'.")
print()

# Verify backend fallback behavior
from services.outcomes import get_graft_survival_1yr as gs_fn
test_city = "Minneapolis"  # Known pancreas program
gs_result = gs_fn("pancreas", test_city)
nat_ps = nat_pancreas.get("national_patient_survival_1yr")
print(f"Backend verification: get_graft_survival_1yr('pancreas', '{test_city}')")
print(f"  Returns: {gs_result}")
print(f"  National patient survival fallback: {nat_ps}")
print(f"  Fallback active: {'YES' if gs_result == nat_ps else 'NO'}")

In [ ]:
# --- 8. Organ-by-Organ Survival Range: 1yr vs 3yr ---
# Show the spread of city-level outcomes for each organ as a paired dot/range plot.
# This highlights intestine's particularly low and variable survival.

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax_idx, (surv_key, surv_label, time_label) in enumerate([
    ("graft_survival_1yr", "Graft Survival", "1-Year"),
    ("patient_survival_1yr", "Patient Survival", "1-Year"),
]):
    ax = axes[ax_idx]
    organ_data = {}
    for organ in ORGANS:
        vals = []
        for city in ALL_CITIES:
            cd = city_outcomes.get(city, {}).get(organ, {})
            v = cd.get(surv_key)
            if v is not None:
                vals.append(v)
        if vals:
            organ_data[organ] = vals

    # Box + strip plot
    plot_organs = [o for o in ORGANS if o in organ_data]
    plot_data = []
    for organ in plot_organs:
        for v in organ_data[organ]:
            plot_data.append({"Organ": organ.title(), "Survival (%)": v})
    plot_df = pd.DataFrame(plot_data)

    if len(plot_df) > 0:
        sns.boxplot(data=plot_df, x="Organ", y="Survival (%)", ax=ax,
                    color="lightsteelblue", fliersize=0, width=0.5)
        sns.stripplot(data=plot_df, x="Organ", y="Survival (%)", ax=ax,
                      color="navy", alpha=0.5, size=5, jitter=0.15)

        # Add national baseline markers
        for j, organ in enumerate(plot_organs):
            nat = outcomes_raw.get(organ, {})
            nat_val = nat.get(f"national_{surv_key}")
            if nat_val is not None:
                ax.plot(j, nat_val, marker="D", color="#d62728", markersize=10,
                        markeredgecolor="white", markeredgewidth=1, zorder=5)

    ax.set_title(f"{time_label} {surv_label} by Organ\n"
                 f"(Strips = individual centers, Red diamond = national baseline)",
                 fontsize=12, fontweight="bold")
    ax.set_ylabel(f"{surv_label} (%)", fontsize=11)
    ax.set_xlabel("")
    ax.set_ylim(60, 102)
    ax.axhline(90, color="gray", linestyle=":", alpha=0.3, linewidth=1)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "04-survival-range-by-organ.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: figures/04-survival-range-by-organ.png")

# Print ranges
print("\nCenter-level survival ranges:")
for organ in ORGANS:
    for key_label, key in [("Graft 1yr", "graft_survival_1yr"),
                           ("Patient 1yr", "patient_survival_1yr")]:
        vals = [city_outcomes.get(c, {}).get(organ, {}).get(key)
                for c in ALL_CITIES]
        vals = [v for v in vals if v is not None]
        if vals:
            print(f"  {organ.title():12s} {key_label:12s}: "
                  f"{min(vals):.1f}% - {max(vals):.1f}%  "
                  f"(range={max(vals)-min(vals):.1f}pp, n={len(vals)} centers)")

In [ ]:
# --- 9. Volume vs Outcomes: Does Center Size Matter? ---
# Scatter n_1yr vs graft_survival_1yr for all organ/city pairs.
# Bayesian shrinkage should pull small centers toward the mean.

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
plot_organs = ["kidney", "liver", "heart", "lung", "pancreas", "intestine"]
organ_colors = sns.color_palette("deep", 6)

for idx, organ in enumerate(plot_organs):
    ax = axes[idx // 3, idx % 3]
    xs, ys, cs, city_labels = [], [], [], []

    for city in ALL_CITIES:
        cd = city_outcomes.get(city, {}).get(organ, {})
        n = cd.get("n_1yr")
        # Use graft survival if available, else patient survival
        gs = cd.get("graft_survival_1yr")
        if gs is None:
            gs = cd.get("patient_survival_1yr")
        rating = cd.get("performance_rating", "as_expected")
        if n is not None and gs is not None:
            xs.append(n)
            ys.append(gs)
            cs.append(RATING_COLORS.get(rating, "#999999"))
            city_labels.append(city)

    if xs:
        ax.scatter(xs, ys, c=cs, s=60, edgecolors="white", linewidth=0.5, zorder=3)

        # National baseline
        nat = outcomes_raw.get(organ, {})
        nat_gs = nat.get("national_graft_survival_1yr")
        if nat_gs is None:
            nat_gs = nat.get("national_patient_survival_1yr")
        if nat_gs is not None:
            ax.axhline(nat_gs, color="gray", linestyle="--", alpha=0.5, linewidth=1)
            ax.text(max(xs) * 0.95, nat_gs + 0.3, "National", fontsize=8,
                    color="gray", ha="right")

        # Label outliers (top 2, bottom 2 by survival)
        paired = sorted(zip(ys, city_labels, xs), key=lambda t: t[0])
        for surv, city, vol in paired[:2] + paired[-2:]:
            ax.annotate(city, (vol, surv), fontsize=7, alpha=0.7,
                        xytext=(5, 5), textcoords="offset points")

    surv_type = "Patient*" if organ == "pancreas" else "Graft"
    ax.set_title(f"{organ.title()}", fontsize=13, fontweight="bold")
    ax.set_xlabel("1-Year Volume (n)" if idx >= 3 else "")
    ax.set_ylabel(f"{surv_type} Survival 1yr (%)" if idx % 3 == 0 else "")

fig.suptitle("Center Volume vs 1-Year Survival\n"
             "(Color = SRTR performance rating: green=better, blue=as expected, "
             "red=worse, gray=insufficient data)",
             fontsize=14, fontweight="bold", y=1.03)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "04-volume-vs-outcomes.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: figures/04-volume-vs-outcomes.png")
print("\nNote: Bayesian shrinkage means small-volume centers are pulled toward")
print("the national mean, which is visible as reduced spread at low volumes.")

## 10. Summary and Limitations

### Key Findings

1. **Kidney and heart** have the highest 1-year graft survival nationally (95.0% and 92.2%), while **intestine** is the most challenging organ (76.3% graft, 82.8% patient survival).

2. **Most centers perform "as expected"** -- the Bayesian hierarchical model with shrinkage toward the national mean means that only centers with strong statistical evidence of deviation are flagged. This is by design: it reduces false alarms from small-volume programs.

3. **The compound success metric** reveals important trade-offs: a city with short wait times but below-average graft survival may rank lower than one with moderate waits and excellent outcomes. This is why TransPlan reports both components transparently.

4. **Pancreas** is the only organ where SRTR does not report adult graft survival separately. TransPlan falls back to patient survival, which is a reasonable proxy (patient death implies graft loss, though not vice versa). All pancreas centers are rated "insufficient data" due to small volumes.

5. **Intestine** programs are rare -- only a handful of TransPlan cities have intestine transplant data, and those that do show the widest survival variation of any organ.

6. **Center volume** shows a modest relationship with outcomes, but the Bayesian shrinkage estimator means small-volume outliers are tempered. The most extreme performance ratings (better/worse) tend to come from centers with sufficient volume to distinguish signal from noise.

### Limitations

| Limitation | Impact | Mitigation |
|-----------|--------|------------|
| Single time point (Jan 2025 PSR) | Does not capture trends in center performance over time | Historical trends tracked in Notebook 03 (SRTR trends module) |
| Bayesian shrinkage may hide real differences | Small-volume centers are pulled toward the mean, potentially masking genuinely exceptional (or poor) programs | Report both point estimate and CI width; flag low-volume centers |
| Pancreas graft survival unavailable | Compound metric uses patient survival as proxy, which overestimates "graft functioning" success | Clearly annotated in API response and UI; pancreas is rare in the model |
| No donor-quality adjustment | Graft survival depends on donor characteristics (DCD vs DBD, donor age) not captured in city-level aggregates | SRTR risk adjustment partially accounts for this; center-level DCD rates could be added |
| Compound metric assumes independence | P(transplant) and P(graft survival) are multiplied as if independent, but center quality may affect both | Conservative simplification; correlation would require joint modeling |
| 22-city subset of US centers | Not all SRTR-reporting centers are included; TransPlan covers major metropolitan transplant hubs | Sufficient for relocation comparison; national coverage deferred to Phase 5 |

### Figures Produced
- `04-national-baselines.png` -- National survival rates by organ (grouped bar chart)
- `04-city-graft-survival-heatmap.png` -- 22 cities x 6 organs heatmap
- `04-kidney-hr-forest-plot.png` -- Kidney hazard ratio forest plot with performance ratings
- `04-compound-success-scatter.png` -- Compound success metric scatter with iso-curves
- `04-performance-rating-distribution.png` -- Stacked bar of SRTR ratings by organ
- `04-survival-range-by-organ.png` -- Box + strip plots of center-level survival spread
- `04-volume-vs-outcomes.png` -- Center volume vs survival scatter by organ